# 02: GCP 実機ハンズオン — BigQuery 管理 Iceberg テーブルで体験する分散ロード・自動スキーマ進化・MERGE・タイムトラベル

## このノートブックで学習できること

- ハンズオン 01（PyIceberg によるローカルメモリ読込）とは異なり、**GCS 上の CSV 読み込みから Iceberg（Parquet + メタデータ）への変換・Upsert までをすべて BigQuery の分散エンジンに任せる**構成
- 1 回目の連携: `id, name, age` の 3 カラム・100 行の CSV を BigQuery 側で分散ロードし、GCS に実体を持つ BigQuery 管理 Iceberg テーブルを作成
- 2 回目の連携: 既存 100 行＋新規 10 行の計 110 行で 3 列目に `prefecture` が追加された CSV（`id, name, prefecture, age`）を、取り込み関数を変更せずそのまま投入し、動的 `ALTER TABLE` ＋ 動的 `MERGE` で自動スキーマ進化＋Upsert を実行
- 3 回目の連携: `prefecture` が欠損した追加 10 行（`id = 111..120`）を同じ関数で投入し、カラムが削除されず欠損列が `NULL` としてロードされることの確認
- BigQuery の `FOR SYSTEM_TIME AS OF` を使い、1〜3 回目の各ロード時点の断面を `SELECT *` でタイムトラベル比較する手順

## 到達ゴール

- 大容量 CSV でも OOM を気にせず処理できる BigQuery 管理 Iceberg テーブルのロード・スキーマ進化・`MERGE` パターンを実装し、PyIceberg 版（01）との違いを説明できる

## 事前準備

詳細は [GCP 実機ハンズオン手順書 02](../docs/handson/02_gcp_bigquery_managed_iceberg_handson.md) を参照してください。事前に以下の 4 点が完了している前提です。

1. `.env` に `GCP_PROJECT_ID`・`GCS_LAKEHOUSE_BUCKET`・`BQ_DATASET`・`BQ_CONNECTION_ID` を設定済み
2. `gcloud auth application-default login`（ADC 認証済み）
3. BigQuery Cloud Resource 接続（`BQ_CONNECTION_ID`）を作成済み
4. 接続のサービスアカウントへ Lakehouse バケットの `roles/storage.objectAdmin` を付与済み


## 1. 環境設定と対象テーブルの初期化

### 1.1 環境変数（`.env`）の読み込みと設定値の確認

リポジトリ直下の `.env` からプロジェクト ID・バケット名・BigQuery データセット・Cloud Resource 接続 ID を読み込みます。データ量は 01 と同じ 100〜120 行に抑えているため、コストは無料枠内（実質 0 円）です。


In [ ]:
from datetime import datetime, timezone
import os
from pathlib import Path
import random
import textwrap
import time

from dotenv import load_dotenv
from google.cloud import bigquery, storage
from google.cloud.exceptions import NotFound
import pandas as pd

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT_DIR / ".env")

PROJECT_ID = os.environ["GCP_PROJECT_ID"]
LOCATION = os.environ.get("GCP_LOCATION", "asia-northeast1")
LAKEHOUSE_BUCKET = os.environ["GCS_LAKEHOUSE_BUCKET"]
LANDING_BUCKET = os.environ.get("GCS_LANDING_BUCKET", LAKEHOUSE_BUCKET)
LANDING_PREFIX = os.environ.get("GCS_LANDING_PREFIX", "handson_schema_evolution/landing")

BQ_DATASET = os.environ["BQ_DATASET"]
BQ_CONNECTION_ID = os.environ.get("BQ_CONNECTION_ID", "biglake-iceberg-conn")
TABLE_NAME = os.environ.get("BQ_MANAGED_TABLE_NAME", "users_master_bq_managed")

CONN_FQN = f"{PROJECT_ID}.{LOCATION}.{BQ_CONNECTION_ID}"
TABLE_REF = f"{PROJECT_ID}.{BQ_DATASET}.{TABLE_NAME}"
TABLE_FQN = f"`{TABLE_REF}`"
STG_REF = f"{PROJECT_ID}.{BQ_DATASET}._stg_{TABLE_NAME}"
STG_FQN = f"`{STG_REF}`"
STORAGE_PREFIX = f"{BQ_DATASET}/{TABLE_NAME}"
STORAGE_URI = f"gs://{LAKEHOUSE_BUCKET}/{STORAGE_PREFIX}"

print(f"LOCATION         : {LOCATION}")
print(f"BQ_CONNECTION_ID : {BQ_CONNECTION_ID}")
print(f"TABLE_NAME       : {TABLE_NAME}")
print(f"STORAGE_PREFIX   : {STORAGE_PREFIX}")


環境変数の読み込みが完了しました。

### 1.2 BigQuery / GCS クライアント初期化と既存テストテーブルのクリーンアップ

何度でもクリーンに再実行できるよう、同名のテーブルおよび GCS 上の旧出力フォルダがあれば削除して初期状態にします。


In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bq_client = bigquery.Client(project=PROJECT_ID, location=LOCATION)


def query_bigquery(sql: str, show_sql: bool = True) -> pd.DataFrame:
    """BigQuery で SQL を実行し、実行クエリを表示したうえで結果を pandas DataFrame で返す。"""
    formatted_sql = textwrap.dedent(sql).strip()
    if show_sql:
        print("--- [BigQuery 実行クエリ] ---")
        print(formatted_sql)
        print("-----------------------------")
    job_config = bigquery.QueryJobConfig(labels={"datacloud": "antigravity"})
    return bq_client.query(formatted_sql, job_config=job_config).to_dataframe()


query_bigquery(f"DROP TABLE IF EXISTS {TABLE_FQN}")
query_bigquery(f"DROP TABLE IF EXISTS {STG_FQN}")

# GCS 上に前回実行のファイルが残っていれば削除して空にする
old_blobs = list(storage_client.list_blobs(LAKEHOUSE_BUCKET, prefix=f"{STORAGE_PREFIX}/"))
for b in old_blobs:
    b.delete()

print(f"初期化完了（GCS 旧ファイル削除: {len(old_blobs)} 件）")


テーブルと GCS 出力先がクリーンな状態になりました。

---

## 2. BigQuery 側で完結する共通取り込み関数の定義

### 2.1 共通関数 `bq_ingest_csv_to_iceberg()` の仕組み

この関数は **ローカル PC のメモリに CSV を 1 バイトも読み込みません**。BigQuery サーバー側で次の 4 ステップを実行します。

1. **分散ロード（ステージング）**: `load_table_from_uri(..., autodetect=True)` で GCS 上の CSV を BigQuery の一時ステージングテーブルへ直接ロード
2. **テーブル作成またはスキーマ進化**: ターゲットの BigQuery 管理 Iceberg テーブルが存在しなければ `CREATE TABLE ... OPTIONS(file_format='PARQUET', table_format='ICEBERG', storage_uri=...)` で作成し、既存ならステージング側にある新カラムを検知して `ALTER TABLE ... ADD COLUMN` を自動実行（逆に欠損列は削除せず維持）
3. **動的 `MERGE`（Upsert＋欠損列 `NULL` 補完）**: ターゲットの全カラムを基準に、ステージングに無い列は `CAST(NULL AS <型>)` で補完する `MERGE` 文を組み立てて分散実行
4. **ステージング削除**: 一時ステージングテーブルを削除


In [ ]:
def bq_ingest_csv_to_iceberg(gcs_uri: str, key_col: str = "id") -> list[str]:
    """GCS の CSV を BigQuery 側で分散ロードし、自動スキーマ進化 + MERGE で Iceberg テーブルへ反映する。"""
    # 1. BigQuery 側で CSV を一時ステージングテーブルへ分散ロード（ローカルメモリ不使用）
    load_cfg = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,
        autodetect=True,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        labels={"datacloud": "antigravity"},
    )
    bq_client.load_table_from_uri(gcs_uri, STG_REF, job_config=load_cfg).result()
    stg_tbl = bq_client.get_table(STG_REF)
    stg_cols = {f.name: f.field_type for f in stg_tbl.schema}
    print(f"[1/4 分散ロード] {gcs_uri} -> ステージング {stg_tbl.num_rows} 行, 列順: {list(stg_cols.keys())}")

    # 2. BigQuery 管理 Iceberg テーブルの作成 または 新カラムの自動追加 (ALTER TABLE)
    try:
        tgt_tbl = bq_client.get_table(TABLE_REF)
        tgt_cols = {f.name: f.field_type for f in tgt_tbl.schema}
        added_cols = []
        for col_name, col_type in stg_cols.items():
            if col_name not in tgt_cols:
                query_bigquery(f"ALTER TABLE {TABLE_FQN} ADD COLUMN `{col_name}` {col_type}")
                added_cols.append(f"{col_name} ({col_type})")
        if added_cols:
            print(f"[2/4 スキーマ進化] ALTER TABLE で新カラムを自動追加しました: {added_cols}")
        else:
            missing = [c for c in tgt_cols if c not in stg_cols]
            if missing:
                print(f"[2/4 スキーマ維持] 入力に欠損列 {missing} がありますが、テーブル定義は維持します")
            else:
                print("[2/4 スキーマ確認] 変更なし")
    except NotFound:
        col_defs = ", ".join(f"`{c}` {t}" for c, t in stg_cols.items())
        query_bigquery(f"""
            CREATE TABLE {TABLE_FQN} ({col_defs})
            WITH CONNECTION `{CONN_FQN}`
            OPTIONS (
              file_format = 'PARQUET',
              table_format = 'ICEBERG',
              storage_uri = '{STORAGE_URI}'
            )
        """)
        print(f"[2/4 作成] BigQuery 管理 Iceberg テーブルを作成しました: {TABLE_NAME}")

    # 3. 動的 MERGE（新カラム反映 + 欠損列は NULL 補完）
    tgt_tbl = bq_client.get_table(TABLE_REF)
    tgt_col_names = [f.name for f in tgt_tbl.schema]
    tgt_col_types = {f.name: f.field_type for f in tgt_tbl.schema}

    select_exprs = [
        f"`{c}`" if c in stg_cols else f"CAST(NULL AS {tgt_col_types[c]}) AS `{c}`"
        for c in tgt_col_names
    ]
    non_key_cols = [c for c in tgt_col_names if c != key_col]
    update_set = ", ".join(f"T.`{c}` = S.`{c}`" for c in non_key_cols)
    insert_cols = ", ".join(f"`{c}`" for c in tgt_col_names)
    insert_vals = ", ".join(f"S.`{c}`" for c in tgt_col_names)

    merge_sql = f"""
        MERGE {TABLE_FQN} T
        USING (SELECT {', '.join(select_exprs)} FROM {STG_FQN}) S
        ON T.`{key_col}` = S.`{key_col}`
        WHEN MATCHED THEN UPDATE SET {update_set}
        WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
    """
    query_bigquery(merge_sql)
    print(f"[3/4 分散 MERGE] Upsert 完了（対象列: {tgt_col_names}）")

    # 4. ステージングテーブルを削除して総行数を確認
    query_bigquery(f"DROP TABLE IF EXISTS {STG_FQN}")
    cnt = query_bigquery(f"SELECT COUNT(*) AS cnt FROM {TABLE_FQN}")["cnt"].iloc[0]
    print(f"[4/4 完了] ステージング削除済み | テーブル総行数={cnt} 行")
    return tgt_col_names


load_checkpoints: dict[str, str] = {}
print("BigQuery 分散取り込み関数 bq_ingest_csv_to_iceberg() を定義しました。")


共通関数の定義が完了しました。ここからは 01 と全く同じ 3 種類の CSV パターンを順に投入して挙動を比較・確認します。

---

## 3. 【1回目】3カラム・100行の CSV を BigQuery で分散ロードする

### 3.1 1回目の CSV（`id, name, age`・100行）を作成して GCS に配置する

テスト用の 100 行 CSV を作成して GCS へアップロードします。


In [ ]:
rng = random.Random(42)

batch1_df = pd.DataFrame({
    "id": list(range(1, 101)),
    "name": [f"User_{i:03d}" for i in range(1, 101)],
    "age": [rng.randint(20, 65) for _ in range(100)],
})

batch1_uri = f"gs://{LANDING_BUCKET}/{LANDING_PREFIX}/bq_batch1_users.csv"
storage_client.bucket(LANDING_BUCKET).blob(f"{LANDING_PREFIX}/bq_batch1_users.csv").upload_from_string(
    batch1_df.to_csv(index=False),
    content_type="text/csv",
)

print(f"1回目 CSV を GCS へアップロードしました: {batch1_uri}")
print(f"行数: {len(batch1_df)} 行 / 列順: {list(batch1_df.columns)}")
display(batch1_df.head(5))


### 3.2 `bq_ingest_csv_to_iceberg()` で Iceberg テーブルを作成・ロードする

BigQuery 側で GCS の CSV を直接読み込み、BigQuery 管理 Iceberg テーブルへ反映します。後段のタイムトラベル検証用に UTC タイムスタンプも記録します。


In [ ]:
bq_ingest_csv_to_iceberg(batch1_uri)

time.sleep(2)
load_checkpoints["1回目 (100行 / 3列)"] = datetime.now(timezone.utc).isoformat()
print(f"記録したタイムスタンプ (UTC): {load_checkpoints['1回目 (100行 / 3列)']}")


### 3.3 BigQuery から 1 回目の結果を確認する

作成した BigQuery 管理 Iceberg テーブルをクエリし、100 行・3 カラムで格納されていることを確認します。


In [ ]:
bq_sample_1 = query_bigquery(f"SELECT * FROM {TABLE_FQN} ORDER BY id LIMIT 5")
print(f"カラム: {list(bq_sample_1.columns)}")
display(bq_sample_1)


100 行・3 カラム（`id, name, age`）が BigQuery 管理 Iceberg テーブルとして作成されました。

---

## 4. 【2回目】3列目に `prefecture` を追加した 110 行の CSV をコード変更なしで連携する

### 4.1 2回目の CSV（`id, name, prefecture, age`・110行）を作成して GCS に配置する

既存 100 行（同じ ID）＋新規 10 行の計 110 行で、3 列目に `prefecture` が挿入された CSV を GCS へアップロードします。


In [ ]:
PREFECTURES = ["東京都", "大阪府", "愛知県", "福岡県", "北海道", "神奈川県", "京都府", "宮城県"]
ages_110 = list(batch1_df["age"]) + [rng.randint(20, 65) for _ in range(10)]

batch2_df = pd.DataFrame({
    "id": list(range(1, 111)),
    "name": [f"User_{i:03d}" for i in range(1, 111)],
    "prefecture": [rng.choice(PREFECTURES) for _ in range(110)],  # 3列目に挿入
    "age": ages_110,
})

batch2_uri = f"gs://{LANDING_BUCKET}/{LANDING_PREFIX}/bq_batch2_users.csv"
storage_client.bucket(LANDING_BUCKET).blob(f"{LANDING_PREFIX}/bq_batch2_users.csv").upload_from_string(
    batch2_df.to_csv(index=False),
    content_type="text/csv",
)

print(f"2回目 CSV を GCS へアップロードしました: {batch2_uri}")
print(f"行数: {len(batch2_df)} 行 / 列順: {list(batch2_df.columns)}")
display(batch2_df.iloc[[0, 1, 99, 100, 109]])


### 4.2 コード変更なしで同じ `bq_ingest_csv_to_iceberg()` を実行する

1 回目と全く同じ関数を呼び出すだけで、BigQuery 側で `ALTER TABLE ... ADD COLUMN prefecture STRING` と分散 `MERGE` が自動実行されます。


In [ ]:
bq_ingest_csv_to_iceberg(batch2_uri)

time.sleep(2)
load_checkpoints["2回目 (110行 / 4列)"] = datetime.now(timezone.utc).isoformat()
print(f"記録したタイムスタンプ (UTC): {load_checkpoints['2回目 (110行 / 4列)']}")


### 4.3 BigQuery から 2 回目の更新結果と都道府県別集計を確認する

既存 100 行が更新され、新規 10 行が追加されて 110 行・4 カラムになっていることを確認します。


In [ ]:
bq_sample_2 = query_bigquery(
    f"SELECT * FROM {TABLE_FQN} WHERE id IN (1, 2, 100, 101, 110) ORDER BY id"
)
bq_by_pref = query_bigquery(
    f"SELECT prefecture, COUNT(*) AS cnt FROM {TABLE_FQN} GROUP BY prefecture ORDER BY cnt DESC"
)

print(f"カラム: {list(bq_sample_2.columns)}")
display(bq_sample_2)
display(bq_by_pref)


3 列目に `prefecture` が挿入された CSV でも `age` を壊すことなく 110 行・4 カラムへ進化・Upsert できました。

---

## 5. 【3回目】`prefecture` カラムが欠損した追加 10 行の CSV を連携する

### 5.1 3回目の CSV（`id = 111..120`・`prefecture` 欠損の 3 列・10 行）を作成して GCS に配置する

01 と同様に、`prefecture` 列が欠損した追加 10 行の CSV を GCS へアップロードします。


In [ ]:
batch3_df = pd.DataFrame({
    "id": list(range(111, 121)),
    "name": [f"User_{i:03d}" for i in range(111, 121)],
    "age": [rng.randint(20, 65) for _ in range(10)],
})

batch3_uri = f"gs://{LANDING_BUCKET}/{LANDING_PREFIX}/bq_batch3_users_missing_pref.csv"
storage_client.bucket(LANDING_BUCKET).blob(f"{LANDING_PREFIX}/bq_batch3_users_missing_pref.csv").upload_from_string(
    batch3_df.to_csv(index=False),
    content_type="text/csv",
)

print(f"3回目 CSV を GCS へアップロードしました: {batch3_uri}")
print(f"行数: {len(batch3_df)} 行 / 列順: {list(batch3_df.columns)}（prefecture 欠損）")
display(batch3_df.head(5))


### 5.2 コード変更なしで同じ `bq_ingest_csv_to_iceberg()` を実行する

動的 `MERGE` が欠損列を `CAST(NULL AS STRING) AS prefecture` で自動補完し、テーブルの `prefecture` 列を削除せず安全にロードします。


In [ ]:
bq_ingest_csv_to_iceberg(batch3_uri)

time.sleep(2)
load_checkpoints["3回目 (120行 / 欠損列NULL)"] = datetime.now(timezone.utc).isoformat()
print(f"記録したタイムスタンプ (UTC): {load_checkpoints['3回目 (120行 / 欠損列NULL)']}")


### 5.3 BigQuery でカラム維持と欠損列の `NULL` ロードを確認する

既存の 110 行には `prefecture` が残ったまま、追加 10 行（`id = 111..120`）のみ `NULL` になっていることを確認します。


In [ ]:
bq_check_3 = query_bigquery(f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(prefecture) AS prefecture_not_null,
  COUNTIF(prefecture IS NULL) AS prefecture_is_null
FROM {TABLE_FQN}
""")
bq_sample_3 = query_bigquery(
    f"SELECT * FROM {TABLE_FQN} WHERE id IN (100, 110, 111, 115, 120) ORDER BY id"
)

print(f"カラム: {list(bq_sample_3.columns)}（prefecture は削除されず維持）")
display(bq_check_3)
display(bq_sample_3)


`total_rows = 120`、`prefecture_not_null = 110`、`prefecture_is_null = 10` となり、欠損列が安全に `NULL` 補完されました。

---

## 6. 各ロード時点の断面を確認する（BigQuery タイムトラベル）

### 6.1 `FOR SYSTEM_TIME AS OF` で 1〜3 回目の断面を `SELECT *` で一括比較する

BigQuery 管理 Iceberg テーブルでは、スキーマ進化前の 1 回目時点であっても **`SELECT *` のまま当時の 3 カラムスキーマでタイムトラベル参照**できます（01 の外部 Iceberg テーブルとの大きな違いの 1 つです）。


In [ ]:
snapshots_summary = []
snapshot_dfs: dict[str, pd.DataFrame] = {}

for label, ts_utc in load_checkpoints.items():
    df_at_t = query_bigquery(
        f"SELECT * FROM {TABLE_FQN} FOR SYSTEM_TIME AS OF TIMESTAMP('{ts_utc}') ORDER BY id"
    )
    snapshot_dfs[label] = df_at_t
    pref_nulls = int(df_at_t["prefecture"].isna().sum()) if "prefecture" in df_at_t.columns else None
    snapshots_summary.append({
        "時点": label,
        "timestamp_utc": ts_utc,
        "行数": len(df_at_t),
        "カラム数": len(df_at_t.columns),
        "カラム一覧": ", ".join(df_at_t.columns),
        "prefecture_NULL件数": pref_nulls,
    })

display(pd.DataFrame(snapshots_summary))


1 回目（100 行・3 列）・2 回目（110 行・4 列・NULL 0 件）・3 回目（120 行・4 列・NULL 10 件）のすべてが `SELECT *` だけでそのまま再現できました。

### 6.2 各時点の末尾レコードを見比べる

各断面の末尾 3 行を並べて確認します。


In [ ]:
for label, df_at_t in snapshot_dfs.items():
    print(f"=== {label} の末尾 3 行 ===")
    display(df_at_t.tail(3))


---

## 7. GCS Lakehouse バケット上の物理ファイルを確認する

BigQuery 管理 Iceberg テーブルであっても、データ実体は自分の GCS バケット上にオープンな Parquet ファイルと Iceberg メタデータとして書き出されています。確認してみましょう。


In [ ]:
blobs = list(storage_client.list_blobs(LAKEHOUSE_BUCKET, prefix=f"{STORAGE_PREFIX}/"))

print(f"GCS 上の物理ファイル総数: {len(blobs)} 件")
for b in blobs:
    rel_path = b.name.replace(f"{STORAGE_PREFIX}/", "")
    print(f"  {rel_path} ({b.size:,} bytes)")


### まとめ: ハンズオン 01（PyIceberg）と 02（BigQuery 管理 Iceberg）の比較

| 観点 | 01: PyIceberg + BigLake REST Catalog | 02: BigQuery 管理 Iceberg テーブル |
| :--- | :--- | :--- |
| **CSV 読込・変換の実行場所** | ローカル / Python ワーカーのメモリ上 | BigQuery の分散エンジン（ローカルメモリ不使用） |
| **向いているデータ規模** | 数 MB〜数百 MB（軽量バッチ・プロトタイプ） | 数 GB〜TB 級の大規模データ |
| **スキーマ進化＋Upsert** | `union_by_name()` ＋ `overwrite()` | ステージング読込 ＋ 動的 `ALTER TABLE` ＋ 動的 `MERGE` |
| **タイムトラベル（`SELECT *`）** | スキーマ進化前時点は列明示が必要 | スキーマ進化前時点も `SELECT *` でそのまま参照可能 |
| **Compaction / GC** | ユーザー側（PyIceberg / Spark）で運用 | BigQuery 側で自動最適化 |
| **外部エンジンからの書き込み** | Spark / Trino / PyIceberg から自由に書き込み可 | 基本は BigQuery が書き込み主体（他エンジンは主に読み取り） |
